In [1]:
const MODEL = "iharm"
const MBH = 6.2e9
const SLOW_LIGHT = true
include("../src/main.jl");

Using model: iharm, change src/set_globals.jl to modify.
Adding slowlight.jl file...


In [2]:
#path to folder with the arxiv name properly indexed
#dump_filepath = "/home/pedro/kharma/iharm3d_out/tmp.00028.h5"
#dump_filepath = "/home/pedro/sample_dump_SANE_a+0.94_MKS_0900.h5"
const all_dumps_path = "/home/pedro/kharma/iharm3d_out/tmp.%05d.h5"

"/home/pedro/kharma/iharm3d_out/tmp.%05d.h5"

In [3]:
const dump_max = 10

params_slowlight = OfSlowLight(dump_max, 0, 0.0, 0.0, 0.0, "")
params_slowlight.current_dumps_path = update_dump_path()

"/home/pedro/kharma/iharm3d_out/tmp.00000.h5"

In [4]:
trat_large = 20. 
const trat_small = 1. 
const beta_crit = 1.0 
const th_beg = 1.74e-2 
const sigma_cut = 1.0 
const sigma_cut_high = -1.0;

In [5]:
const params = read_header(params_slowlight.current_dumps_path);

Initializing grid from: /home/pedro/kharma/iharm3d_out/tmp.00000.h5


custom electron model loaded from dump file...
Using mixed tp_over_te with trat_small = 1, trat_large = 20, and beta_crit = 1
Using Funky Modified Kerr-Schild coordinates FMKS
MKS parameters a: 0.937500 hslope: 0.300000 Rin: 1.001876 Rout: 1000.000000
FMKS parameters poly_xt: 0.820000 poly_alpha: 14.000000 mks_smooth: 0.500000 poly_norm: 0.757817
Grid start (startx): 1.874000951149755e-03, 0.000000000000000e+00, 0.000000000000000e+00 stop (stopx): 6.907755278982137e+00, 1.000000000000000e+00, 6.283185307179586e+00
grid dx: 5.395219748461709e-02, 7.812500000000000e-03, 6.283185307179586e+00


In [6]:
const simulation_data = Vector{IharmData}(undef, 3)

# everytime you load a file slow_light mode will automatically advance to the next one
simulation_data[1] = load_data(params_slowlight.current_dumps_path, trat_large)
simulation_data[2] = load_data(params_slowlight.current_dumps_path, trat_large)
simulation_data[3] = load_data(params_slowlight.current_dumps_path, trat_large)

params_slowlight.tA = simulation_data[1].t;
params_slowlight.tB = simulation_data[2].t;

params_slowlight.tf = get_specific_dump_time(params_slowlight.dump_max);

Loading data from '/home/pedro/kharma/iharm3d_out/tmp.00000.h5' into 'iharm' module...
All primitives successfully loaded. Dimensions: (128, 128, 1)
Loading data from '/home/pedro/kharma/iharm3d_out/tmp.00001.h5' into 'iharm' module...
All primitives successfully loaded. Dimensions: (128, 128, 1)
Loading data from '/home/pedro/kharma/iharm3d_out/tmp.00002.h5' into 'iharm' module...
All primitives successfully loaded. Dimensions: (128, 128, 1)


In [7]:
# Observer distance in gravitational radii (Rg)
const ro = 1000.0

# Inclination angle (deg) — angle between the observer and the BH spin axis
const th = 163.0

# Azimuthal angle (deg) — rotation around the system
const phi = 0.0

# Image resolution — total geodesics traced = res^2
const res = 1
const pixels_x = 1
const pixels_y = 1

# Distance to the source (in parsecs, converted to code units)
const SourceD = 16.9e6 * PC

# Radius where ray integration stops
const Rstop = 100.0

# Event horizon radius for a Kerr black hole
const Rh = 1 + sqrt(1. - params.a * params.a)

# Observing frequency (Hz), e.g. 230 GHz for EHT-like images
const freq = 230e9

# Image plane size (in Rg), scaled from physical distance
const DXsize = SourceD / L_unit / MUAS_PER_RAD * 160
const DYsize = SourceD / L_unit / MUAS_PER_RAD * 160

# Field of view (radians)
const fovx = DXsize / ro
const fovy = DYsize / ro

# Image offsets (can be used to shift the camera)
const xoff = 0.0
const yoff = 0.0


0.0

In [8]:
# Calculate the camera position in native coordinates
Xcamera = MVec4(camera_position(ro, th, phi, params.a, params.Rout))

# Unitless frequency 
const freq_unitless = freq * HPL / (ME * CL * CL)

# Array that will hold the Intensity value for each pixel
Image = zeros(Float64, pixels_x, pixels_y)
midplane_crossings = zeros(Int, pixels_x, pixels_y)
nsteps = zeros(Int, pixels_x, pixels_y)

# Number of threads used in the calculation
const nthreads = Threads.nthreads() + 1

println("Allocating workspaces for $nthreads threads...")
# Number of maximum steps in the geodesic calculation
const maxnstep = 15000

# Allocating the scratchpad vector for each thread
thread_trajs = [Vector{OfTrajM}(undef, maxnstep) for _ in 1:nthreads]
for t in 1:nthreads
    for k in 1:maxnstep
        thread_trajs[t][k] = OfTrajM(
            0.0, 
            MVec4(undef), MVec4(undef), MVec4(undef), MVec4(undef)
        )
    end
end

# This will hold the exact number of steps for each pixel.
const all_geodesics = Matrix{Vector{OfTrajM}}(undef, pixels_x, pixels_y)

# Allocate an array to hold the minimum time found by EACH thread
# Initialized to 0.0 because your photon times will be negative
thread_t0 = zeros(Float64, nthreads)

# tgeoi needs to find the maximum (closest to zero), so initialize very negative
thread_tgeoi = fill(-1e100, nthreads) 

# tgeof needs to find the minimum (most negative), so initialize at zero
thread_tgeof = zeros(Float64, nthreads)

p = Progress(
    pixels_x * pixels_y; 
    desc = "Raytracing Image...", 
    showspeed = true, 
    barlen = 30
)
ProgressMeter.ijulia_behavior(:clear)

println("Tracing Geodesics...")
Threads.@threads for i in 0:(pixels_x - 1)
    tid = Threads.threadid() 
    
    for j in 0:(pixels_y - 1)
        nstep, midplane_crossings[i+1,j+1] = get_pixel(
            thread_trajs[tid], i, j, Xcamera, 
            fovx, fovy, freq_unitless, 
            pixels_x, pixels_y, params.a, 
            Rh, params.Rout, Rstop, xoff, yoff
        ) 
        nsteps[i+1, j+1] = nstep
        # Save to permanent storage
        all_geodesics[i + 1, j + 1] = thread_trajs[tid][1:nstep]

        final_step_time = thread_trajs[tid][nstep].X[1]
        if final_step_time < thread_t0[tid]
            thread_t0[tid] = final_step_time
        end

        # tgeoi and tgeof calculations following how ipole does it
        pixel_tgeoi = 1.0
        pixel_tgeof = 1.0
        for k in 1:nstep
            X = thread_trajs[tid][k].X
            K = thread_trajs[tid][k].Kcon
        
            log_r = X[2]          # must be log(r)
            t_coord = X[1]
            k_r = K[2]            # must match Kcon[1]
        
            if pixel_tgeoi > 0.0 && log_r < log(100.0)
                pixel_tgeoi = t_coord
            end
        
            if pixel_tgeof > 0.0 && log_r > log(100.0) && k_r < 0.0
                pixel_tgeof = t_coord
            end
        end
        final_step_time = thread_trajs[tid][nstep].X[1]
        
        if pixel_tgeoi < 0.0 && pixel_tgeoi > thread_tgeoi[tid]
            thread_tgeoi[tid] = pixel_tgeoi
        end
        if pixel_tgeof < 0.0 && pixel_tgeof < thread_tgeof[tid]
            thread_tgeof[tid] = pixel_tgeof
        elseif pixel_tgeof > 0.0 && final_step_time < thread_tgeof[tid]
            thread_tgeof[tid] = final_step_time
        end
        
        ProgressMeter.next!(
                p; 
                showvalues = [
                    (:thread_id, tid), 
                    (:pixel, "($i, $j)"), 
                    (:total_done, "$(i*pixels_y + j)/$(pixels_x * pixels_y)")
            ]
        )
    end
end

Image *= freq^3
finish!(p);

t0 = minimum(thread_t0)
tgeof = minimum(thread_tgeof) # The most negative time in the active zone (Oldest file needed)
tgeoi = maximum(thread_tgeoi) # The least negative time in the active zone (Newest file needed)

println("Calculated t0 (absolute longest time): $t0")
println("Calculated tgeof (oldest active time): $tgeof")
println("Calculated tgeoi (newest active time): $tgeoi")

# Eliminate arrays from RAM
thread_to = nothing
thread_tgeoi = nothing
thread_tgeof = nothing
thread_trajs = nothing
GC.gc()

Allocating workspaces for 9 threads...
Tracing Geodesics...
Calculated t0 (absolute longest time): -1096.384946339862
Calculated tgeof (oldest active time): -1096.384946339862
Calculated tgeoi (newest active time): -910.1325351226865


In [9]:
#print images every 10M
ImageCadence = 10; 
#So the earliest image we can form is at time
last_img_target = params_slowlight.tA - tgeof
nimgs_concurrently = round(Int, 2 + abs(t0)/ImageCadence)
#MovieArray = zeros(OfImg, pixels_x, pixels_y, nimgs_concurrently);
MovieArray = [zero(OfImg) for _ in 1:pixels_x, _ in 1:pixels_y, _ in 1:nimgs_concurrently]
#times to produce movies
target_times = zeros(Float64, nimgs_concurrently)

#can we actually produce this image with the snapshots we have?
valid_images = zeros(Float64, nimgs_concurrently)

println("First Image will be produced at $last_img_target")
nimg = 1;

# how many images are still doing?
nopenimgs = 1;

output = "Image.%05d.txt"

First Image will be produced at 2196.3876268372587


"Image.%05d.txt"

In [14]:
using DelimitedFiles
while true
    while (last_img_target + t0 < params_slowlight.tB)
        target_times[nimg] = last_img_target
        if(last_img_target + tgeoi < params_slowlight.tf - params.rmax_geo)
            valid_images[nimg] = 1;
            nopenimgs += 1
            for i in 1:pixels_x
                for j in 1:pixels_y
                    MovieArray[i, j, nimg].nstep = nsteps[i, j]
                    MovieArray[i, j, nimg].Intensity = 0.0
                    MovieArray[i, j, nimg].tau = 0.0
                    MovieArray[i, j, nimg].tauF = 0.0
                end
            end
            nimg += 1
            if nimg > nimgs_concurrently
                nimg = 1
            end
        end
        last_img_target += ImageCadence;
    end

    for k in 1:nimgs_concurrently
        if valid_images[k] == 0
            continue
        end
        do_output = true

        for i in 1:pixels_x
            for j in 1:pixels_y
                if j == 1
                    print("$i")
                end
                Xi = MVec4(undef)
                Kconi = MVec4(undef)
                Xf = MVec4(undef)
                Kconf = MVec4(undef)
                Xhalf = MVec4(undef)
                Kconhalf = MVec4(undef)
                traj = all_geodesics[i, j]
                nstep = copy(MovieArray[i,j,k].nstep)
                while(nstep > 2)
                    for a in 1:NDIM
                        Xi[a] = traj[nstep].X[a]
                        Xhalf[a] = traj[nstep].Xhalf[a]
                        Xf[a] = traj[nstep - 1].X[a]
                        Kconi[a] = traj[nstep].Kcon[a]
                        Kconhalf[a] = traj[nstep].Kconhalf[a]
                        Kconf[a] = traj[nstep - 1].Kcon[a]
                    end
                    Xi[1] += target_times[k] + 1e-5
                    Xhalf[1] += target_times[k] + 1.e-5;
                    Xf[1] += target_times[k] + 1.e-5

                    if(Xi[1] < params_slowlight.tA)
                        Xf[1] += params_slowlight.tA - Xi[1]
                        Xhalf[1] += params_slowlight.tA - Xi[1]
                        Xi[1] = params_slowlight.tA;
                    end
                    if(Xi[1] >= params_slowlight.tB)
                        if(Xf[1] >= params_slowlight.tf)
                            Xi[1] += params_slowlight.tf - Xf[1];
                            Xhalf[1] += params_slowlight.tf - Xf[1];
                            Xf[1] = params_slowlight.tf;
                        else
                            break
                        end
                    end
                    ji, ki = get_jk(Xi, Kconi, freq, params.a, simulation_data)
                    jf, kf = get_jk(Xf, Kconf, freq, params.a, simulation_data)

                    MovieArray[i,j,k].Intensity = approximate_solve(MovieArray[i,j,k].Intensity, ji, ki, jf, kf, traj[nstep - 1].dl)
                    
                    if(abs(params_slowlight.tA - 1200) < 1 && MovieArray[i,j,k].Intensity != 0)
                        println("This happened with $k, $i, $j, nstep = $nstep, tA = $(params_slowlight.tA)")
                        println("Intensity is $(MovieArray[i,j,k].Intensity) with $ji, $ki, $jf, $kf, dl = $(traj[nstep - 1].dl)")
                        println("$Xi, $Xf")
                        error()
                    end
                    nstep -= 1
                end
                MovieArray[i,j,k].nstep = copy(nstep)
                if (nstep != 2)
                    do_output = false
                end
            end
        end
        if(do_output)
            Image = map(x -> x.Intensity, MovieArray[:,:,k]) .* freq^3
            file_name = Printf.format(Printf.Format(output), target_times[k])
            writedlm(file_name, Image)
            println("Saving image $(file_name)")
            valid_images[k] = 0
            nopenimgs -= 1
        end 
    end
    if (nopenimgs <= 1)
        break
    end
    update_data!(simulation_data)
end

1This happened with 1, 1, 1, nstep = 409, tA = 1200.0065090072605
Intensity is 8.208550415530874e-48 with 0.0, 0.0, 0.0, 0.0, dl = 685.569657740246
[1200.1666909971375, 2.991882391672987, 0.9618592801575537, -0.12546852532577088], [1200.3763878373957, 3.000459418098996, 0.9617958615011094, -0.12433339264384057]


LoadError: 

In [11]:
println("Generating plots...")

using CairoMakie

# Coordinate Conversions
d_kpc = 16.9
d_cm = d_kpc * 3.086e21            
fov_rg = fovx
half_fov_rg = fov_rg / 2

theta_rad = (half_fov_rg * L_unit) / d_cm   
theta_μas = theta_rad * MUAS_PER_RAD       
xlims = (-theta_μas, theta_μas)
ylims = (-theta_μas, theta_μas)

Ny, Nx = size(Image)
x = range(xlims[1], xlims[2], length=Nx)
y = range(ylims[1], ylims[2], length=Ny)

# Define your zoom limits
zoom_lims = (-25.0, 25.0)

# Figure Setup
fig = Figure(size = (1200, 500))

# --- Left Panel: Intensity ---
ax_img = Axis(fig[1, 1],
    xlabel = "Relative R.A [μas]",
    ylabel = "Relative Dec [μas]",
    xlabelsize = 24,
    ylabelsize = 24,
    xticklabelsize = 20,
    yticklabelsize = 20,
    # Lock limits to -30 to 30
    limits = (zoom_lims, zoom_lims),
    aspect = DataAspect()
)

# Safe Extrema (Crash fix for entirely black images)
cmin, cmax = extrema(Image)
if cmin == cmax
    cmax = cmin + 1e-5
end

hm_img = heatmap!(ax_img, x, y, Image;
    colormap = :hot,
    colorrange = (cmin, cmax)
)

Colorbar(fig[1, 2], hm_img;
    label = "Intensity",
    labelsize = 20,
    ticklabelsize = 16,
    width = 15
)

# --- Right Panel: Crossings ---
ax_cross = Axis(fig[1, 3],
    xlabel = "Relative R.A [μas]",
    xlabelsize = 24,
    xticklabelsize = 20,
    yticklabelsize = 20,
    # Lock limits to -30 to 30
    limits = (zoom_lims, zoom_lims),
    aspect = DataAspect()
)

# Colormap logic: Use a reversed magma map. 
# n=0 will be very light, and higher n will get progressively darker/sharper.
max_cross = maximum(midplane_crossings)
n_levels = min(15, max(2, max_cross + 1)) # +1 to ensure we count 0 as a level
discrete_colormap = cgrad(:magma, n_levels, categorical = true, rev = true)

cmin_cross = 0
cmax_cross = max(1, max_cross)

hm_cross = heatmap!(ax_cross, x, y, midplane_crossings;
    colormap = discrete_colormap,
    colorrange = (cmin_cross, cmax_cross)
)

tick_step = max_cross > 15 ? ceil(Int, max_cross / 5) : 1

Colorbar(fig[1, 4], hm_cross;
    label = "Number of Crossings",
    labelsize = 20,
    ticklabelsize = 16,
    width = 15,
    ticks = 0:tick_step:cmax_cross
)

colgap!(fig.layout, 2, 40)

# Display the final plot
fig

Generating plots...


LoadError: InterruptException: